In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame
from listUtils import getFlatList
from musicdb import PanDBIO
from sys import prefix
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource']


# DB-Specific

In [3]:
from lib import setlistfm
mio   = setlistfm.MusicDBIO(verbose=False, mkDirs=False)
apiio = setlistfm.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant SetListFM DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/SetListFM


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
masterArtists      = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
masterArtistsData  = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtistsData".format(db.lower()))
searchArtists      = mio.data.getSearchArtistData()
knownArtists       = Series(dtype='object') #mio.data.getSummaryNameData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Global Master Search:      {0}".format(len(masterArtists.get())))
print("   Global Master Search Data: {0}".format(len(masterArtistsData.get())))
print("   Search Artists:            {0}".format(len(searchArtists)))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

SetListFM Search Results
   Global Master Search:      23504
   Global Master Search Data: 1400
   Search Artists:            22104
   Errors:                    85
   Known Summary IDs:         0


# Search For New Artists

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=False,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [22]:
from musicdb import PanDBIO
from gate import IOStore

pdbio = PanDBIO()
mmeDF = pdbio.getData().sort_values(by=["Counts", "Albums"], ascending=False)

ios     = IOStore()
mdbio   = ios.get(db="MusicBrainz")
refData = mdbio.data.getSummaryRefData().to_dict()

mbIDData = mmeDF[mmeDF["MusicBrainz"].notna()][["ArtistName", "MusicBrainz"]]
mbIDData["MBRef"] = mbIDData["MusicBrainz"].apply(refData.get).apply(lambda x: x.split('/')[-1] if isinstance(x,str) else None)

searchedForMasterArtists = masterArtists.get()
artistNamesToGet = mbIDData[~mbIDData["MusicBrainz"].isin(searchedForMasterArtists.keys())]

print("{0} Search Results".format(db))
print("   Known Master Artist Names:    {0}".format(mbIDData.shape[0]))
print("   Known Spotify Artist Names:   {0}".format(len(searchedForMasterArtists)))
print("   Artist Names To Get:          {0}".format(len(artistNamesToGet)))

del pdbio
del mmeDF
del refData
del mbIDData

#   Artist Names To Get:          793373

SetListFM Search Results
   Known Master Artist Names:    814451
   Known Spotify Artist Names:   27519
   Artist Names To Get:          786853


## Download Artist Searches

In [23]:
from timeutils import Timestat, TermTime

## Run @ 3-4 PM every day

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "11:50pm")
n    = 0
maxN = 1400

searchedForMasterArtistsData = masterArtistsData.get()
searchedForMasterArtists     = masterArtists.get()
searchedForErrors            = errors.get()
nErr = []
for i,(idx,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["ArtistName"]
    artistID = row["MusicBrainz"]
    mbID = row["MBRef"]
    if searchedForMasterArtists.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoResults(artistName=artistName, mbID=mbID)
    if response is None:
        print("Error ==> {0}".format((artistID,mbID,artistName)))
        searchedForErrors[artistID] = True
        apiio.sleep(15)
        nErr.append(artistID)
        if len(nErr) >= 6:
            break
        continue

    nErr = []
    searchedForMasterArtistsData[artistID] = response
    searchedForMasterArtists[artistID] = artistName
    apiio.sleep(20)
    n += 1
        
    if n % 5 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForMasterArtists), db))
        masterArtists.save(data=searchedForMasterArtists)
        masterArtistsData.save(data=searchedForMasterArtistsData)
        errors.save(data=searchedForErrors)
        print("="*150)
        apiio.wait(10)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
            
ts.stop()
print("Saving [{0} / {1}] {2} Searched For Artist IDs".format(len(searchedForMasterArtists), len(searchedForMasterArtistsData), db))
masterArtists.save(data=searchedForMasterArtists)
masterArtistsData.save(data=searchedForMasterArtistsData)
if len(nErr) > 0:
    errors.save(data=searchedForErrors)
    for artistID in nErr:
        print("del searchedForErrors['{0}']".format(artistID))
    print("errors.save(data=searchedForErrors)")

Process [Getting SetListFM ArtistIDs] Start    ==> Time Is 2022-04-23 10:47:28
========================= termTime(day=tomorrow,time=11:00am) =========================
   ====> Terminate Time Set To 2022-04-24 11:00:00 <====
   ====> Will Terminate Process 12 Minutes and 31 Seconds From Now
Searching For Bap Kennedy (ecc71d06-dde1-4db4-8fd3-0d2e3dd2e4b1)                              True
Searching For Maria Schneider (a0eb374d-ce13-4900-b8aa-a25fde420355)                          True
Searching For Naragonia (dd871e13-7460-455a-bd53-1af0944fb38b)                                True
Searching For St. Petersburg Philharmonic Orchestra (241e0792-a99a-4a97-9eb6-ad88eb748d2f)    True
Searching For Wake Owl (e749f971-63c8-4869-936d-fbc0f5e9acb7)                                 True
5/?        : Process [Getting SetListFM ArtistIDs] Has Run For 1 Minute and 45 Seconds.
Saving 27524 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For Gregory Page (c92849e8-2c3a-4c2b-ab28-2fb5da593031)                             True
Searching For Keaton (a0ae5279-1588-429b-9b15-70efb457e0e3)                                   True
Searching For Trigger the Bloodshed (8d5660c7-50ab-43d5-84f9-79624b1f39ef)                    True
Searching For John Popper (d73c9a5d-5d7d-47ec-b15a-a924a1a271c4)                              True
Searching For Roger Norrington (2403f8c6-8ccc-48d6-977f-de0baa2d6fed)                         True
10/?       : Process [Getting SetListFM ArtistIDs] Has Run For 3 Minutes and 38 Seconds.
Saving 27529 SetListFM Searched For Artist IDs


Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

Searching For Futurebirds (e432ac6d-fc48-436a-bbee-3f112902f6ae)                              True
Searching For KJ Sawka (4da511d4-4dfe-4389-9c0a-4ecf35cb0467)                                 True
Searching For Francesca Lombardo (eb719427-1347-440c-8def-1d30e89c9517)                       True
Searching For Howard Ashman (7a45c6ea-bb66-4e85-8643-271658fe935e)                            True
Searching For The Deer Tracks (dfcf365b-1a07-4627-b54e-637e916363e6)                          True
15/?       : Process [Getting SetListFM ArtistIDs] Has Run For 5 Minutes and 31 Seconds.
Saving 27534 SetListFM Searched For Artist IDs


...

==> Error in SetListFM search for Compact Disk Dummies
Error ==> ('100289869541857586668960815138953544847', 'ef13e544-e35f-483e-9616-53dd7c6367a4', 'Compact Disk Dummies')
Searching For Lorenz Büffel (5c5c7c27-5b9a-468b-a9ab-18b05d86c25d)                           

HTTP exception: HTTPError('429 Client Error: Too Many Requests for url: https://api.setlist.fm/rest/1.0/artist/5c5c7c27-5b9a-468b-a9ab-18b05d86c25d')


==> Error in SetListFM search for Lorenz Büffel
Error ==> ('155129401683092282537119538701124448454', '5c5c7c27-5b9a-468b-a9ab-18b05d86c25d', 'Lorenz Büffel')
Searching For Daniele Liverani (ba5ebfcb-5134-4f69-8ca1-49db8f9ddd71)                         

HTTP exception: HTTPError('429 Client Error: Too Many Requests for url: https://api.setlist.fm/rest/1.0/artist/ba5ebfcb-5134-4f69-8ca1-49db8f9ddd71')


==> Error in SetListFM search for Daniele Liverani
Error ==> ('30043587855655597963313294794338118145', 'ba5ebfcb-5134-4f69-8ca1-49db8f9ddd71', 'Daniele Liverani')
Process [Getting SetListFM ArtistIDs] Ran For 7 Hours and 22 Minutes    ==> Time Is 2022-04-23 18:10:12
Saving [28671 / 2552] SetListFM Searched For Artist IDs
del searchedForErrors['260523060702005307194169312891497960966']
del searchedForErrors['204688363120401809827902113017497819567']
del searchedForErrors['52601397499669132316615570594880471136']
del searchedForErrors['100289869541857586668960815138953544847']
del searchedForErrors['155129401683092282537119538701124448454']
del searchedForErrors['30043587855655597963313294794338118145']
errors.save(data=searchedForErrors)


In [24]:

del searchedForErrors['260523060702005307194169312891497960966']
del searchedForErrors['204688363120401809827902113017497819567']
del searchedForErrors['52601397499669132316615570594880471136']
del searchedForErrors['100289869541857586668960815138953544847']
del searchedForErrors['155129401683092282537119538701124448454']
del searchedForErrors['30043587855655597963313294794338118145']
errors.save(data=searchedForErrors)

## Save Results

In [25]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(mad):
    df = None
    if isinstance(mad,dict) and len(mad) > 0:
        df = Series(mad).apply(Series)
        #df = DataFrame({v['mbid']: {k2: v2 for k2,v2 in v.items() if k2 not in []} for k,v in mad.items()}).T
    return df
        
def getResponseDataFrame(mad):
    df = getArtistNamesDataFrame(mad)
    if not isinstance(df,DataFrame):
        return None
    return df

In [26]:
mad = masterArtistsData.get()
df  = getResponseDataFrame(mad)

if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = setlistfm.MusicDBIO(local=False).data.getSearchArtistData()
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Results".format(searchDF.shape[0]))
    searchDF = searchDF.drop_duplicates(keep='first')
    print("Found {0} Unique Results".format(searchDF.shape[0]))
    print("Saving Data")
    setlistfm.MusicDBIO(local=False).data.saveSearchArtistData(data=searchDF)

Found 2552 New Artists
Found 26119 Previous Artists
Found 28671 Total Results
Found 28671 Unique Results
Saving Data


In [27]:
masterArtistsData.save(data={})

# Download Artist Data

In [ ]:
mio   = setlistfm.MusicDBIO(verbose=False,local=True,mkDirs=False)
apiio = setlistfm.RawAPIData(debug=False)

## Find Artists To Download

In [ ]:
artistData = {}
for searchTerm,searchResults in searchArtists.iteritems():
    if isinstance(searchResults,list):
        artistData.update({item["id"]: item for item in searchResults if isinstance(item,dict)})
artistData       = DataFrame(artistData).T.sort_values(by='id')
artistNames      = artistData[["name", "url"]]
localArtistsDict = localArtists.get()
artistIDsToGet   = artistNames[~artistNames.index.isin(localArtistsDict.keys())].sample(frac=1)

print("{0} Search Results".format(db))
print("   Available IDs:      {0}".format(len(artistNames)))
print("   Known Artist IDs:   {0}".format(len(localArtistsDict)))
print("   Artist IDs To Get:  {0}".format(len(artistIDsToGet)))

## Download The Data

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Artist Data".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "11:00am")
tt = TermTime("today", "9:00pm")
#tt = TermTime("today", "12:05pm")
maxN = 50000000

n  = 0
localArtistsDict     = localArtists.get()
searchedForErrors    = errors.get()
for i,(artistID,row) in enumerate(artistIDsToGet.iterrows()):
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue

    artistName = row["name"]
    artistURL  = row["url"]

    dbID   = artistID
    modVal = mio.mv.get(dbID)
    if mio.data.getRawArtistInfoFilename(modVal, dbID).exists():
        localArtistsDict[artistID] = artistName
        continue
    try:
        response = apiio.getArtistInfoResults(artistName=artistName, artistURL=artistURL)
    except:
        response = None
    if response is None:
        print("Error ==> {0}".format(artistName))
        searchedForErrors[artistID] = True
        apiio.sleep(3.5)
        continue
    
    localArtistsDict[artistID] = artistName
    mio.data.saveRawArtistInfoData(data=response, modval=modVal, dbID=dbID)
    apiio.sleep(6.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
mio.data.getRawArtistInfoData(mio.mv.get(3540473060), 3540473060)

In [ ]:
localArtistsDict

In [ ]:
localArtists.save(data=localArtistsDict)


In [ ]:
tracks
disc_count